# Quickstart

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [3]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

100.0%
100.0%
100.0%
100.0%


In [4]:
batch_size = 64

train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

In [5]:
for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [6]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [7]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, X):
        X = self.flatten(X)
        logits = self.linear_relu_stack(X)
        return logits

In [8]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [9]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [10]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

In [11]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100 * correct):>0.1f}%, Avg_loss: {test_loss:>8f} \n")

In [12]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t + 1}\n------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)

Epoch 1
------------------------------
loss: 2.295022 [   64/60000]
loss: 2.286556 [ 6464/60000]
loss: 2.271185 [12864/60000]
loss: 2.271611 [19264/60000]
loss: 2.254993 [25664/60000]
loss: 2.208836 [32064/60000]
loss: 2.237888 [38464/60000]
loss: 2.189246 [44864/60000]
loss: 2.195156 [51264/60000]
loss: 2.169490 [57664/60000]
Test Error: 
 Accuracy: 30.3%, Avg_loss: 2.158808 

Epoch 2
------------------------------
loss: 2.167770 [   64/60000]
loss: 2.159248 [ 6464/60000]
loss: 2.108192 [12864/60000]
loss: 2.126896 [19264/60000]
loss: 2.080716 [25664/60000]
loss: 2.004874 [32064/60000]
loss: 2.053672 [38464/60000]
loss: 1.965861 [44864/60000]
loss: 1.977399 [51264/60000]
loss: 1.906330 [57664/60000]
Test Error: 
 Accuracy: 59.1%, Avg_loss: 1.901414 

Epoch 3
------------------------------
loss: 1.936977 [   64/60000]
loss: 1.903598 [ 6464/60000]
loss: 1.795256 [12864/60000]
loss: 1.830757 [19264/60000]
loss: 1.727176 [25664/60000]
loss: 1.666350 [32064/60000]
loss: 1.700175 [38464/600

In [13]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


In [14]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

In [15]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"


# Tensors

In [57]:
import torch
import numpy as np

In [20]:
data = [[1, 2], [3, 4]]
x_data = torch.tensor(data)
x_data

tensor([[1, 2],
        [3, 4]])

In [21]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)
x_np

tensor([[1, 2],
        [3, 4]])

In [22]:
x_ones = torch.ones_like(x_data)
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float)
print(f"Random Tensor: \n {x_rand} \n")

Ones Tensor: 
 tensor([[1, 1],
        [1, 1]]) 

Random Tensor: 
 tensor([[0.8395, 0.3106],
        [0.2940, 0.6840]]) 



In [28]:
shape = (3, 3, 2)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor} \n")

Random Tensor: 
 tensor([[[0.3353, 0.8867],
         [0.8894, 0.9006],
         [0.0580, 0.3903]],

        [[0.2845, 0.8975],
         [0.8265, 0.9229],
         [0.0353, 0.3509]],

        [[0.3845, 0.9313],
         [0.8292, 0.8601],
         [0.7318, 0.7677]]]) 

Ones Tensor: 
 tensor([[[1., 1.],
         [1., 1.],
         [1., 1.]],

        [[1., 1.],
         [1., 1.],
         [1., 1.]],

        [[1., 1.],
         [1., 1.],
         [1., 1.]]]) 

Zeros Tensor: 
 tensor([[[0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.]]]) 



In [29]:
tensor = torch.rand(3, 4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


In [30]:
if torch.accelerator.is_available():
    tensor = tensor.to(torch.accelerator.current_accelerator())

In [46]:
tensor = torch.rand(4, 4)
print(tensor)
print(tensor[0])
print(tensor[:, 0])
print(tensor[..., -1])

tensor([[0.5163, 0.2387, 0.2939, 0.7751],
        [0.4150, 0.9507, 0.0934, 0.3479],
        [0.3189, 0.6636, 0.9503, 0.8993],
        [0.2126, 0.9399, 0.8908, 0.1027]])
tensor([0.5163, 0.2387, 0.2939, 0.7751])
tensor([0.5163, 0.4150, 0.3189, 0.2126])
tensor([0.7751, 0.3479, 0.8993, 0.1027])


In [47]:
(tensor.T @ tensor).shape
print(tensor * tensor.T)
print(tensor @ tensor.T)

tensor([[0.2666, 0.0991, 0.0937, 0.1648],
        [0.0991, 0.9038, 0.0620, 0.3270],
        [0.0937, 0.0620, 0.9031, 0.8011],
        [0.1648, 0.3270, 0.8011, 0.0105]])
tensor([[1.0106, 0.7383, 1.2993, 0.6755],
        [0.7383, 1.2058, 1.1649, 1.1007],
        [1.2993, 1.1649, 2.2539, 1.6304],
        [0.6755, 1.1007, 1.6304, 1.7326]])


In [48]:
agg = tensor.sum()
agg_item = agg.item()
print(agg_item, type(agg_item))

8.609010696411133 <class 'float'>


In [49]:
agg, type(agg)

(tensor(8.6090), torch.Tensor)

In [50]:
x = torch.zeros(4, 4)
x.add_(1)
print(x)

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])


In [54]:
n = x.numpy()
n

array([[1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.]], dtype=float32)

In [52]:
x

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

In [55]:
x.add_(5)
print(x)

tensor([[6., 6., 6., 6.],
        [6., 6., 6., 6.],
        [6., 6., 6., 6.],
        [6., 6., 6., 6.]])


In [56]:
print(n)

[[6. 6. 6. 6.]
 [6. 6. 6. 6.]
 [6. 6. 6. 6.]
 [6. 6. 6. 6.]]


In [59]:
x = np.zeros((4, 4))
y = torch.from_numpy(x)

In [60]:
x

array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

In [61]:
y

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]], dtype=torch.float64)

In [62]:
x += 1
x

array([[1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.]])

In [63]:
y

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]], dtype=torch.float64)

In [64]:
y.device

device(type='cpu')

# Datasets & DataLoaders

In [67]:

import pandas as pd
from torchvision.io import decode_image
from torch.utils.data import Dataset

In [68]:
class CustomImageDataset(Dataset):
    def __init__(self, annotation_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotation_file)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = decode_image(img_path)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)

        return image, label

# Transforms

# Build the Neural Network

In [1]:
import os
import torch
from torch import nn

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [4]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
x = torch.rand(1, 28, 28, device=device)
logits = model(x)
print(logits)

tensor([[-5.5697e-03, -6.8846e-02,  8.1354e-02,  2.2383e-05, -7.9273e-02,
         -8.5248e-02, -5.8218e-02,  4.2231e-03, -4.9097e-02, -5.3960e-02]],
       grad_fn=<AddmmBackward0>)


In [6]:
logits.shape

torch.Size([1, 10])

In [10]:
pred_prob = nn.functional.softmax(dim=1, input=logits)
print(pred_prob)

tensor([[0.1025, 0.0962, 0.1118, 0.1031, 0.0952, 0.0946, 0.0972, 0.1035, 0.0981,
         0.0977]], grad_fn=<SoftmaxBackward0>)


In [19]:
pred_prob.argmax(1)

tensor([2])

In [20]:
input_image = torch.rand(3, 28, 28)
print(input_image.size())

torch.Size([3, 28, 28])


In [27]:
flatten = nn.Flatten()
x = flatten(input_image)
print(x)
print(x.shape)

tensor([[0.4627, 0.1036, 0.1347,  ..., 0.5920, 0.0507, 0.5595],
        [0.3636, 0.1556, 0.1637,  ..., 0.8538, 0.5199, 0.0855],
        [0.9733, 0.5288, 0.7525,  ..., 0.3014, 0.9997, 0.4618]])
torch.Size([3, 784])


In [31]:
layer1 = nn.Linear(in_features=28 * 28, out_features=20)
x = layer1(x)
print(x)
print(x.shape)

tensor([[ 0.2548,  0.4155,  0.4471,  0.2617, -0.3472, -0.0464,  0.1812, -0.0527,
          0.2579, -0.8417, -0.2375, -0.5150,  0.3214, -0.1135, -0.2026, -0.8735,
         -0.2648,  0.3729,  0.2116,  0.2171],
        [ 0.3772,  0.0511,  0.5514,  0.0073, -0.1840, -0.0684,  0.1876, -0.1692,
          0.2349, -0.9834, -0.2794, -0.3110,  0.2479,  0.0045, -0.2922, -0.3033,
         -0.4299,  0.3899,  0.4850,  0.3373],
        [ 0.5152,  0.2353,  0.4734,  0.2798, -0.1462, -0.3591,  0.0894,  0.2398,
          0.4061, -0.8383, -0.6062, -0.4998,  0.0886, -0.2766, -0.4022, -0.7278,
         -0.5185,  0.0377,  0.3219,  0.3160]], grad_fn=<AddmmBackward0>)
torch.Size([3, 20])


In [32]:
print(layer1)

Linear(in_features=784, out_features=20, bias=True)


In [34]:
layer1.weight.shape

torch.Size([20, 784])

In [36]:
layer1.bias.shape

torch.Size([20])

In [37]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0240, -0.0329,  0.0352,  ...,  0.0223,  0.0228, -0.0331],
        [ 0.0064,  0.0157,  0.0136,  ..., -0.0117,  0.0050, -0.0066]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([-0.0007, -0.0223], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0021,  0.0317, -0.0399,  ..., -0.0047, -0.0180,  0.0405],
        [-0.0341,  0.0426,  0.0360,  ..., -0.0345,  0.0020,  0.0293]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | 

# Automatic Differentiation with torch.autograd

In [38]:
import torch

x = torch.ones(5)  # input tensor
y = torch.zeros(3)  # expected output
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w) + b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

In [39]:
print(z.shape)
print(loss.shape)

torch.Size([3])
torch.Size([])


In [40]:
print(z)

tensor([-2.5912, -3.8122,  0.2624], grad_fn=<AddBackward0>)


In [41]:
print(loss)

tensor(0.3090, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)


In [42]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x000001D88CE03310>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x000001D88CE03310>


In [43]:
loss.backward()

In [44]:
print(w.grad)
print(b.grad)

tensor([[0.0232, 0.0072, 0.1884],
        [0.0232, 0.0072, 0.1884],
        [0.0232, 0.0072, 0.1884],
        [0.0232, 0.0072, 0.1884],
        [0.0232, 0.0072, 0.1884]])
tensor([0.0232, 0.0072, 0.1884])


In [48]:
print(loss)

tensor(0.3090, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)


In [47]:
print(w)

tensor([[-1.6550, -0.4720, -0.7008],
        [-1.1846, -1.5698,  0.5587],
        [-0.9265,  1.3606,  0.9131],
        [ 1.4493, -0.4678,  0.0424],
        [-0.5812, -1.0540, -0.2050]], requires_grad=True)


In [45]:
print(x)

tensor([1., 1., 1., 1., 1.])


In [46]:
print(y)

tensor([0., 0., 0.])


In [49]:
z = torch.matmul(x, w) + b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w) + b
print(z.requires_grad)

True
False


In [50]:
inp = torch.eye(4, 5, requires_grad=True)
out = (inp+1).pow(2).t()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n{inp.grad}")
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")
inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

First call
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])

Second call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])

Call after zeroing gradients
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])


In [51]:
print(inp)

tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.]], requires_grad=True)


In [52]:
print(out)

tensor([[4., 1., 1., 1.],
        [1., 4., 1., 1.],
        [1., 1., 4., 1.],
        [1., 1., 1., 4.],
        [1., 1., 1., 1.]], grad_fn=<TBackward0>)


# Optimizing Model Parameters